# Agreement Comparison: Humans vs. Weak / Synthetic Labels

Quantifies inter-annotator agreement on binary `safe`/`harmful` and compares it
to two non-human label sources on the same axis:

1. **Human – Human** — pairwise across duplicate-prompt annotations.
2. **Human – WildGuardMix** — human annotator vs. the original WildGuardMix `prompt_harm_label`.
3. **Human – Qwen3-8B (synthetic)** — human annotator vs. labels from `RedHatAI/Qwen3-8B-quantized.w4a16` (`data/harm_labels.csv`).

All comparisons use binary `{safe, harmful}` so they are directly comparable.
Reports percent agreement and Cohen's $\kappa$ for each.

In [1]:
import pandas as pd
import numpy as np
from itertools import combinations
from sklearn.metrics import cohen_kappa_score

## Load annotations and synthetic labels

In [2]:
ann = pd.read_parquet('../../data/annotations/intent_annotations.parquet')
qwen = pd.read_csv('../../data/harm_labels.csv').rename(columns={'id': 'ID', 'harm_label': 'qwen_harm'})
print('annotations:', ann.shape)
print('qwen subset:', qwen.shape)

annotations: (1730, 8)
qwen subset: (173, 2)


## Binary collapse

Four-point annotator labels collapse: `Completely Safe`, `Uncertain Safe` $\rightarrow$ `safe`;
`Uncertain Harmful`, `Completely Harmful` $\rightarrow$ `harmful`. `Flag for Removal` is dropped.
WildGuardMix labels (`harmful`/`unharmful`) and Qwen synthetic labels use the same collapse.

In [3]:
def to_binary(label):
    s = str(label).strip().lower()
    if s in ('completely harmful', 'uncertain harmful', 'harmful'):
        return 'harmful'
    if s in ('completely safe', 'uncertain safe', 'unharmful', 'safe'):
        return 'safe'
    return None  # Flag for Removal / NaN

ann['annotator_binary'] = ann['Annotator Harm'].map(to_binary)
ann['dataset_binary']   = ann['Dataset Harm'].map(to_binary)

clean = ann[ann['annotator_binary'].notna()].copy()
print(f'Total rows: {len(ann)}, after dropping Flag-for-Removal: {len(clean)}')

Total rows: 1730, after dropping Flag-for-Removal: 1724


## Cohen's $\kappa$ on binary labels

$$\kappa = \frac{p_o - p_e}{1 - p_e}$$

where $p_o$ is observed pairwise agreement and $p_e$ is the agreement expected from
the raters' marginal class distributions. $\kappa = 1$ is perfect agreement;
$\kappa = 0$ is chance-level given the marginals.

### 1. Human – Human (pairwise across duplicate prompts)

For each prompt with $\geq 2$ annotators, we take every unordered pair of
annotator labels and run Cohen's $\kappa$ on the two parallel lists.
Pairwise Cohen's (rather than Fleiss' $\kappa$) is used because duplicate group
sizes vary across prompts.

In [4]:
prompt_groups = clean.groupby('Prompt')['annotator_binary'].apply(list)
duplicate_groups = [labels for labels in prompt_groups if len(labels) >= 2]

pair_a, pair_b = [], []
for labels in duplicate_groups:
    for x, y in combinations(labels, 2):
        pair_a.append(x); pair_b.append(y)

n_prompts = len(duplicate_groups)
n_pairs   = len(pair_a)
hh_agree = np.mean([a == b for a, b in zip(pair_a, pair_b)])
hh_kappa = cohen_kappa_score(pair_a, pair_b)

print(f'Duplicate prompts: {n_prompts}, pairwise comparisons: {n_pairs}')
print(f'  % agreement: {hh_agree:.3f}')
print(f"  Cohen's κ:   {hh_kappa:.3f}")

Duplicate prompts: 225, pairwise comparisons: 945
  % agreement: 0.777
  Cohen's κ:   0.548


### 2. Human – WildGuardMix weak labels (full annotated set)

In [5]:
wg = clean.dropna(subset=['dataset_binary']).copy()
hw_agree = (wg['annotator_binary'] == wg['dataset_binary']).mean()
hw_kappa = cohen_kappa_score(wg['annotator_binary'], wg['dataset_binary'])
print(f'n = {len(wg)}')
print(f'  % agreement: {hw_agree:.3f}')
print(f"  Cohen's κ:   {hw_kappa:.3f}")

n = 1724
  % agreement: 0.725
  Cohen's κ:   0.450


### 3. Human – Qwen3-8B synthetic labels (subset where Qwen ran)

In [6]:
merged = clean.merge(qwen, on='ID', how='inner')
merged['qwen_binary'] = merged['qwen_harm'].map(to_binary)
merged = merged.dropna(subset=['qwen_binary'])

hq_agree = (merged['annotator_binary'] == merged['qwen_binary']).mean()
hq_kappa = cohen_kappa_score(merged['annotator_binary'], merged['qwen_binary'])
print(f'n = {len(merged)}')
print(f'  % agreement: {hq_agree:.3f}')
print(f"  Cohen's κ:   {hq_kappa:.3f}")

n = 172
  % agreement: 0.529
  Cohen's κ:   0.035


## Same-subset sanity check

Qwen labels exist only for ~172 prompts. Recompute the other two comparisons
restricted to that subset so all three rows are apples-to-apples.

In [7]:
# Human - WildGuard, restricted to Qwen subset
sub_hw_agree = (merged['annotator_binary'] == merged['dataset_binary']).mean()
sub_hw_kappa = cohen_kappa_score(merged['annotator_binary'], merged['dataset_binary'])
print(f'Human - WildGuard (Qwen subset, n={len(merged)}):')
print(f'  % agreement: {sub_hw_agree:.3f} | κ: {sub_hw_kappa:.3f}')

# Human - Human, restricted to duplicates that fall in the Qwen subset
sub_prompts = set(merged['Prompt'])
sub_dups = [labels for prompt, labels in prompt_groups.items()
            if prompt in sub_prompts and len(labels) >= 2]
sub_a, sub_b = [], []
for labels in sub_dups:
    for x, y in combinations(labels, 2):
        sub_a.append(x); sub_b.append(y)
if sub_a:
    print(f'Human - Human (Qwen subset, {len(sub_dups)} prompts, {len(sub_a)} pairs):')
    print(f'  % agreement: {np.mean([a == b for a, b in zip(sub_a, sub_b)]):.3f} | κ: {cohen_kappa_score(sub_a, sub_b):.3f}')

Human - WildGuard (Qwen subset, n=172):
  % agreement: 0.744 | κ: 0.488
Human - Human (Qwen subset, 54 prompts, 449 pairs):
  % agreement: 0.757 | κ: 0.507


## Summary table

In [8]:
summary = pd.DataFrame([
    {'comparison': 'Human – Human',         'n': n_pairs,    'pct_agree': hh_agree, 'cohen_kappa': hh_kappa},
    {'comparison': 'Human – WildGuardMix',  'n': len(wg),    'pct_agree': hw_agree, 'cohen_kappa': hw_kappa},
    {'comparison': 'Human – Qwen3-8B',      'n': len(merged),'pct_agree': hq_agree, 'cohen_kappa': hq_kappa},
])
summary

,comparison,n,pct_agree,cohen_kappa
0,Human – Human,945,0.776720,0.548123
1,Human – WildGuardMix,1724,0.725058,0.450389
2,Human – Qwen3-8B,172,0.529070,0.035447
